# Yeoubi: supervised fine-tuning (SFT)

Fine-tunes a pretrained Yeoubi checkpoint (by default the decay branch's `final.pt`) to answer
prompts. Training sees `prompt + "\nAssistant:\n" + response`; only the response is masked
and learned. Runs on Colab (T4) or Kaggle (GPU T4 x2, Internet on) with an `HF_TOKEN` secret
that has write access.

1. **Build the mix** (first run only; reused afterwards from `SFT_DATA_REPO`): smol-smoltalk,
   Dolly, SQuAD, OASST2, GSM8K and short NuminaMath problems, each row at most `MAX_BYTES`
   bytes. A small `TEACHER` model can add short answers; it is off by default.
2. **Fine-tune** with `--training_stage reasoning_sft --finetune_from` the base checkpoint,
   self-conditioning on, A8 throughout. Math rows (GSM8K, NuminaMath) use the
   `Reasoning: ... Final answer: ...` format. Checkpoints go to `SFT_REPO`; a disconnected
   run resumes from there.
3. **Correction stage** (on by default): a short run that plants wrong bytes in answers and
   teaches the model to fix them, so the sampler can revise its own draft.
4. **Try it**: sample a few prompts from the most finished model.

Run all in every session: finished stages are skipped, an unfinished one resumes.

The teacher writes answers to prompts; it cannot teach through its output probabilities,
because Yeoubi predicts bytes and the teacher predicts tokens, so the two do not line up.

In [ ]:
# SPDX-License-Identifier: BigCode-OpenRAIL-M-1.0
REPO = "https://github.com/Fury7425/Yeoubi.git"
BASE_REPO = ""        # model repo with the checkpoint to fine-tune, such as "your-name/yeoubi-decay-1"
BASE_FILE = "final.pt"
SFT_DATA_REPO = ""    # dataset repo for the built mix, such as "your-name/yeoubi-sft-data"
SFT_REPO = ""         # model repo for this run's checkpoints, such as "your-name/yeoubi-sft-1"
MAX_BYTES = 3500      # prompt + separator + response; the model's window is 4096 bytes and
                      # longer rows are dropped, never cut, so no answer stops mid-sentence
TEACHER = ""          # off: the mix is public datasets only. To add short teacher answers,
                      # set "LiquidAI/LFM3-1.7B", or "Qwen/Qwen3.5-2B" (Apache-2.0, needs
                      # vLLM's nightly build: set VLLM_PIP below)
VLLM_PIP = "vllm"     # for Qwen3.5: "vllm --extra-index-url https://wheels.vllm.ai/nightly"
TEACHER_ROWS = 20000  # prompts the teacher answers; about 15-25 min on a T4
EPOCHS = 1            # passes over the mix; sets max_steps, so keep it fixed for one run
SELF_COND = True      # adds the self-conditioning layer (zero-initialized, so it starts
                      # as a no-op); costs about 20% more time per step
CORRECTION = True     # second, short stage after SFT (--training_stage correction): random
                      # wrong bytes are planted in the answer and the model learns to fix
                      # them, which is what lets the sampler revise with --revise_threshold
CORRECTION_FRACTION = 0.1   # its length as a share of the SFT steps
CORR_REPO = ""        # its checkpoints; "" means SFT_REPO + "-correction"
KEEP = 2              # resumable checkpoints kept in SFT_REPO
HOURS = 11            # train.py stops itself after this and saves a resumable checkpoint

In [ ]:
import glob
import hashlib
import json
import math
import os
import random
import re
import subprocess
import sys
import threading
import time

assert BASE_REPO and SFT_DATA_REPO and SFT_REPO, "set BASE_REPO, SFT_DATA_REPO and SFT_REPO"
WORK = "/tmp"   # /tmp on both hosts: Kaggle copies /kaggle/working out as notebook output
if not os.path.isdir(f"{WORK}/Yeoubi"):
    !git clone --depth 1 {REPO} {WORK}/Yeoubi
%cd {WORK}/Yeoubi
!pip -q install -r requirements.txt
!nvidia-smi -L

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("no HF_TOKEN secret: the repo calls below will fail")

from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
api.create_repo(SFT_DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
api.create_repo(SFT_REPO, private=True, exist_ok=True)
CORR_REPO = CORR_REPO or SFT_REPO + "-correction"
if CORRECTION:
    api.create_repo(CORR_REPO, private=True, exist_ok=True)
# The sync thread uploads from LOCAL_CKPT to TARGET_REPO; the correction cell repoints both.
LOCAL_CKPT, TARGET_REPO = f"{WORK}/ckpt", SFT_REPO
os.makedirs(LOCAL_CKPT, exist_ok=True)
SEP = "\nAssistant:\n"   # TrainConfig.assistant_separator

In [ ]:
# The teacher script, run as its own process by the build cell above when TEACHER is set.
TEACHER_SCRIPT = r"""
import json, re, sys
from vllm import LLM, SamplingParams

model, src, dst = sys.argv[1:4]
prompts = [json.loads(line)["prompt"] for line in open(src, encoding="utf-8")]
# T4 has no bf16; vLLM computes these bf16 checkpoints in fp16.
llm = LLM(model=model, dtype="half", max_model_len=2048, gpu_memory_utilization=0.85)
tok = llm.get_tokenizer()
SYSTEM = ("You are a helpful assistant. Answer clearly and briefly in plain English, "
          "in at most 120 words. Do not show your reasoning.")
chats = [tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                  {"role": "user", "content": p}],
                                 tokenize=False, add_generation_prompt=True,
                                 enable_thinking=False)
         for p in prompts]
outs = llm.generate(chats, SamplingParams(temperature=0.3, top_p=0.9, max_tokens=220))
kept = 0
with open(dst, "w", encoding="utf-8") as f:
    for p, o in zip(prompts, outs):
        c = o.outputs[0]
        # LFM3 thinks by default and enable_thinking=False is not always honoured: strip a
        # closed trace, drop anything left unclosed or cut off at max_tokens.
        text = re.sub(r"<think>.*?</think>", "", c.text, flags=re.S).strip()
        if c.finish_reason != "stop" or not text or "<think>" in text:
            continue
        f.write(json.dumps({"prompt": p, "response": text}, ensure_ascii=False) + "\n")
        kept += 1
print(f"teacher kept {kept} of {len(prompts)} answers")
"""

In [ ]:
# Builds the mix once and stores it in SFT_DATA_REPO; later sessions skip straight to training.
BUILT = "train.jsonl" in api.list_repo_files(SFT_DATA_REPO, repo_type="dataset")
if BUILT:
    print("mix already built in", SFT_DATA_REPO)
else:
    from datasets import load_dataset

    rows, seen, counts = [], set(), {}
    rng = random.Random(0)

    def add(src, prompt, response=None, reasoning=None, answer=None, cap=None):
        """Keep a row if it fits MAX_BYTES and its prompt is new; True when kept."""
        if cap is not None and counts.get(src, 0) >= cap:
            return False
        prompt = (prompt or "").strip()
        if response is not None:
            body, rec = response.strip(), {"prompt": prompt, "response": response.strip()}
        else:
            # data.py renders reasoning rows as "Reasoning:\n...\nFinal answer:\n...".
            body = f"Reasoning:\n{reasoning.strip()}\nFinal answer:\n{answer.strip()}"
            rec = {"prompt": prompt, "reasoning": reasoning.strip(), "answer": answer.strip()}
        if not prompt or not body or len((prompt + SEP + body).encode("utf-8")) > MAX_BYTES:
            return False
        key = hashlib.sha1(prompt.lower().encode("utf-8")).hexdigest()
        if key in seen:
            return False
        seen.add(key)
        rows.append({"source": src, **rec})
        counts[src] = counts.get(src, 0) + 1
        return True

    # smol-smoltalk: HF's SFT set for its 135M/360M models; first user -> assistant turn.
    # The openhermes slice is left out: it was generated by GPT-4. Prompts past the cap
    # become the teacher's prompts, so teacher and dataset rows never share a prompt.
    teacher_prompts = []
    for ex in load_dataset("HuggingFaceTB/smol-smoltalk", split="train", streaming=True):
        if "openhermes" in ex.get("source", ""):
            continue
        msgs = [m for m in ex["messages"] if m["role"] in ("user", "assistant")]
        if len(msgs) < 2 or msgs[0]["role"] != "user" or msgs[1]["role"] != "assistant":
            continue
        if counts.get("smoltalk", 0) < 60000:
            add("smoltalk", msgs[0]["content"], msgs[1]["content"])
        elif TEACHER and len(teacher_prompts) < TEACHER_ROWS:
            if len(msgs[0]["content"].encode("utf-8")) <= 1500:
                teacher_prompts.append(msgs[0]["content"].strip())
        else:
            break

    # Dolly: human-written; instruction plus optional reference text.
    for ex in load_dataset("databricks/databricks-dolly-15k", split="train"):
        ctx = (ex["context"] or "").strip()
        add("dolly", ex["instruction"] + (f"\n\n{ctx}" if ctx else ""), ex["response"])

    # SQuAD: answer from a given passage; short exact answers.
    squad = load_dataset("rajpurkar/squad", split="train").shuffle(seed=0)
    for ex in squad:
        if counts.get("squad", 0) >= 15000:
            break
        add("squad", f"{ex['context']}\n\nQuestion: {ex['question']}", ex["answers"]["text"][0])

    # OASST2: human conversations stored as trees; English root prompt -> best-ranked reply.
    oa = load_dataset("OpenAssistant/oasst2", split="train")
    by_id = {m["message_id"]: m for m in oa}
    for m in oa:
        if m["role"] == "assistant" and m["lang"] == "en" and m["rank"] == 0:
            parent = by_id.get(m["parent_id"])
            if parent and parent["role"] == "prompter" and parent["parent_id"] is None:
                add("oasst2", parent["text"], m["text"], cap=8000)

    # GSM8K: human-written grade-school math; calculator annotations <<...>> removed.
    for ex in load_dataset("openai/gsm8k", "main", split="train"):
        steps, _, final = ex["answer"].partition("####")
        add("gsm8k", ex["question"], reasoning=re.sub(r"<<[^>]*>>", "", steps).strip(), answer=final.strip())

    # NuminaMath-CoT: easier sources only; long solutions fall out on MAX_BYTES. Stored in
    # the reasoning format like GSM8K: the worked solution, then its last \boxed{} value.
    easy = {"orca_math", "synthetic_math", "cn_k12"}
    boxed = re.compile(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}")
    for ex in load_dataset("AI-MO/NuminaMath-CoT", split="train", streaming=True):
        if counts.get("numina", 0) >= 15000:
            break
        found = boxed.findall(ex["solution"]) if ex["source"] in easy else []
        if found:
            add("numina", ex["problem"], reasoning=ex["solution"], answer=found[-1])

    if TEACHER and teacher_prompts:
        with open(f"{WORK}/teacher_prompts.jsonl", "w", encoding="utf-8") as f:
            for p in teacher_prompts:
                f.write(json.dumps({"prompt": p}) + "\n")
        with open(f"{WORK}/teacher_gen.py", "w", encoding="utf-8") as f:
            f.write(TEACHER_SCRIPT)
        # A separate process, so vLLM hands the GPU back before training starts.
        subprocess.run(f"{sys.executable} -m pip install -q {VLLM_PIP}", shell=True, check=True)
        subprocess.run([sys.executable, f"{WORK}/teacher_gen.py", TEACHER,
                        f"{WORK}/teacher_prompts.jsonl", f"{WORK}/teacher.jsonl"], check=True)
        for line in open(f"{WORK}/teacher.jsonl", encoding="utf-8"):
            r = json.loads(line)
            add("teacher", r["prompt"], r["response"])

    rng.shuffle(rows)
    n_val = max(200, len(rows) // 100)
    for name, part in (("val.jsonl", rows[:n_val]), ("train.jsonl", rows[n_val:])):
        with open(f"{WORK}/{name}", "w", encoding="utf-8") as f:
            for r in part:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    meta = {"train": len(rows) - n_val, "val": n_val, "sources": counts,
            "max_bytes": MAX_BYTES, "teacher": TEACHER or None}
    with open(f"{WORK}/sft_meta.json", "w") as f:
        json.dump(meta, f, indent=1)
    for name in ("val.jsonl", "train.jsonl", "sft_meta.json"):
        api.upload_file(path_or_fileobj=f"{WORK}/{name}", path_in_repo=name,
                        repo_id=SFT_DATA_REPO, repo_type="dataset")
    print(json.dumps(meta, indent=1))

In [ ]:
def step_of(path):
    match = re.search(r"step_(\d+)", os.path.basename(path))
    return int(match.group(1)) if match else -1


def is_resumable(name):
    return name.startswith("step_") and name.endswith(".pt") and "pre_a4" not in name


def newest_in(repo):
    return max(filter(is_resumable, api.list_repo_files(repo)), key=step_of, default="")


train_file = hf_hub_download(SFT_DATA_REPO, "train.jsonl", repo_type="dataset")
val_file = hf_hub_download(SFT_DATA_REPO, "val.jsonl", repo_type="dataset")
meta = json.load(open(hf_hub_download(SFT_DATA_REPO, "sft_meta.json", repo_type="dataset")))
# 16 examples per optimizer step, whatever the GPU count (see the train cell).
max_steps = math.ceil(meta["train"] * EPOCHS / 16)

# --finetune_from loads weights only and starts at step 0 with a fresh optimizer; after a
# disconnect the run continues from its own checkpoint with --resume_from instead.
newest = newest_in(SFT_REPO)
if newest:
    init = ["--resume_from", hf_hub_download(SFT_REPO, newest)]
else:
    init = ["--finetune_from", hf_hub_download(BASE_REPO, BASE_FILE)]
print(f"{meta['train']} train rows, {max_steps} steps; start: {newest or BASE_REPO + '/' + BASE_FILE}")

In [ ]:
_sizes = {}
_sync_lock = threading.Lock()   # the background loop and the final sync must not overlap


def sync_checkpoints(stable_only=True):
    """Upload finished checkpoints from LOCAL_CKPT to TARGET_REPO; drop old ones there."""
    with _sync_lock:
        uploaded = False
        for path in sorted(glob.glob(f"{LOCAL_CKPT}/*.pt"), key=step_of):
            size = os.path.getsize(path)
            # torch.save writes in place, so a file is only known complete once its
            # size has stopped changing between two polls.
            if stable_only and _sizes.get(path) != size:
                _sizes[path] = size
                continue
            api.upload_file(path_or_fileobj=path, path_in_repo=os.path.basename(path), repo_id=TARGET_REPO)
            os.remove(path)   # only after the upload committed
            uploaded = True
        if not uploaded:
            return
        resumable = sorted(filter(is_resumable, api.list_repo_files(TARGET_REPO)), key=step_of)
        for old in resumable[:-KEEP]:
            api.delete_file(old, TARGET_REPO)
        if len(resumable) > KEEP:
            # Deleted files keep counting toward storage until the history is squashed.
            api.super_squash_history(TARGET_REPO)


def _sync_loop():
    while True:
        time.sleep(60)
        try:
            sync_checkpoints()
        except Exception as error:   # a network hiccup must not end the syncing
            print("checkpoint sync failed, will retry:", error)


if "_sync_thread" not in globals():
    _sync_thread = threading.Thread(target=_sync_loop, daemon=True)
    _sync_thread.start()

In [ ]:
import torch

# Model and batch flags match the pretraining run (see colab_train.ipynb). SFT-specific:
# --lr 5e-5 / --muon_lr 0.005: a quarter of the pretraining peak, a common SFT starting point,
#   not tuned for this model; 200 warmup steps, then cosine down to 10%.
# --a4_warmup_fraction 0: stays A8 like the base checkpoint.
# --use_self_cond: must stay the same for every session of one run.
GPUS = torch.cuda.device_count()
assert GPUS in (1, 2, 4), f"16 examples per step do not split evenly over {GPUS} GPUs"
ACCUM = 4 // GPUS
INIT = " ".join(f'"{a}"' for a in init)
COMMON = (f'--use_self_cond {str(SELF_COND).lower()} --train_data "{train_file}" --val_data "{val_file}" '
          f"--a4_warmup_fraction 0 --max_seq_len 4096 --batch_size 4 --grad_accum_steps {ACCUM} "
          "--hidden_dim 768 --n_heads 12 --ffn_dim 2048 --bf16 false --compile_model false "
          "--gradient_checkpointing false --save_every 500 --val_every 500 --muon_weight_decay 0.01 "
          f'--max_runtime_hours {HOURS} --num_workers 2 --wandb_project ""')
if "final.pt" in api.list_repo_files(SFT_REPO):
    print("SFT already finished:", SFT_REPO)
else:
    get_ipython().system(
        f"PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node {GPUS} train.py "
        f"--training_stage reasoning_sft {INIT} --output_dir {LOCAL_CKPT} "
        f"--max_steps {max_steps} --warmup_steps 200 --lr 5e-5 --muon_lr 0.005 --min_lr_ratio 0.1 {COMMON}")
    sync_checkpoints(stable_only=False)   # train.py has exited, so every file is complete

In [ ]:
# Correction stage: starts from the finished SFT model. In each example 15% of the answer
# bytes are swapped for random ones (--correction_noise_prob) and the noise level is capped
# at 0.5 (--correction_t_max), so the model learns to find and fix wrong visible bytes as
# well as fill masked ones. Lower LR than SFT: it is a short adjustment, not new learning.
# The loss switches to uniform weighting automatically in this stage.
SFT_DONE = "final.pt" in api.list_repo_files(SFT_REPO)
if not CORRECTION:
    print("correction stage off")
elif not SFT_DONE:
    print("SFT has not finished yet: run this notebook again in the next session")
elif "final.pt" in api.list_repo_files(CORR_REPO):
    print("correction already finished:", CORR_REPO)
else:
    sync_checkpoints(stable_only=False)
    LOCAL_CKPT, TARGET_REPO = f"{WORK}/ckpt_corr", CORR_REPO
    os.makedirs(LOCAL_CKPT, exist_ok=True)
    corr_steps = max(200, math.ceil(max_steps * CORRECTION_FRACTION))
    newest_corr = newest_in(CORR_REPO)
    corr_init = (["--resume_from", hf_hub_download(CORR_REPO, newest_corr)] if newest_corr
                 else ["--finetune_from", hf_hub_download(SFT_REPO, "final.pt")])
    CORR_INIT = " ".join(f'"{a}"' for a in corr_init)
    print(f"correction: {corr_steps} steps from {newest_corr or SFT_REPO + '/final.pt'}")
    get_ipython().system(
        f"PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node {GPUS} train.py "
        f"--training_stage correction {CORR_INIT} --output_dir {LOCAL_CKPT} "
        f"--max_steps {corr_steps} --warmup_steps 50 --lr 2e-5 --muon_lr 0.002 --min_lr_ratio 0.1 "
        f"--correction_noise_prob 0.15 --correction_t_max 0.5 {COMMON}")
    sync_checkpoints(stable_only=False)

In [ ]:
# Samples from the most finished model: the correction stage's final.pt (with revision on),
# else SFT's final.pt, else SFT's newest checkpoint. Sampler settings from the sweep on the
# base model: ReMDM remasking, temperature 0.45, top-p 0.9. --revise_threshold 0.9 lets the
# sampler overwrite a visible byte the model is 90% sure is wrong; it only makes sense on a
# checkpoint trained with the correction stage.
revise = []
if CORRECTION and "final.pt" in api.list_repo_files(CORR_REPO):
    ckpt, revise = hf_hub_download(CORR_REPO, "final.pt"), ["--revise_threshold", "0.9", "--revise_t_max", "0.5"]
elif "final.pt" in api.list_repo_files(SFT_REPO):
    ckpt = hf_hub_download(SFT_REPO, "final.pt")
else:
    ckpt = hf_hub_download(SFT_REPO, newest_in(SFT_REPO))
print("sampling", ckpt, "with revision" if revise else "")
PROMPTS = ["What is water made of?", "Give me three tips for learning a new language.",
           "Tom has 3 apples and buys 5 more. How many apples does he have?",
           "Write a short, friendly greeting for a new classmate."]
for q in PROMPTS:
    out = subprocess.run([sys.executable, "sample.py", "--checkpoint", ckpt, "--prompt", q + SEP,
                          "--block", "--block_size", "64", "--length", "512", "--auto_length",
                          "--steps", "32", "--remask", "remdm", "--temperature", "0.45", "--top_p", "0.9"] + revise,
                         capture_output=True, text=True)
    print("=" * 60, "\n" + (out.stdout.strip() or out.stderr[-2000:]))